# Tables

In [3]:
import pandas as pd
import os
from table import Table
import json
import matplotlib.pyplot as plt
import numpy as np

import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import matplotlib.colors as mcolors

In [4]:
#datasets = ['connect-4', 'dry-bean', 'hand_digits', 'chess', 'shuttle'] #, 'connect-4', 'poker_hand']
datasets = ['T1B', 'T2'] #, 'CIFAR10', 'CIFAR100coarse'] 
table_ae = Table(name='mrae')

In [5]:
def get_val_loss(file_path):
    """Read the JSON file and get the value of 'val_loss'."""
    with open(file_path, 'r') as file:
        data = json.load(file)
        return data.get('val_loss', float('inf'))  # Returns infinity if 'val_loss' is not present

def compare_files(directory):
    files = os.listdir(directory)

    files_dict = {}
    
    for file_name in files:
        if file_name.endswith('_dropout') and file_name.find('_RAE'):
            base_name = file_name[:-8] #Removes '_dropout'
            suffix = 'dropout'
        #else:
        elif file_name.find('_RAE'):
            base_name = file_name
            suffix = 'no_dropout'
        if base_name not in files_dict:
            files_dict[base_name] = {}
        files_dict[base_name][suffix] = os.path.join(directory, file_name)
    
    # Compare files in pairs
    best_files = []

    for base_name, file_group in files_dict.items():
        """file_do0 = file_group['no_dropout']
        file_do1 = file_group['dropout']
            
        val_loss_do0 = get_val_loss(file_do0)
        val_loss_do1 = get_val_loss(file_do1)
            
        if val_loss_do0 < val_loss_do1:
            best_files.append(file_do0)
        else:
            best_files.append(file_do1)"""
        file_do0 = file_group.get('no_dropout')
        file_do1 = file_group.get('dropout')

        # si falta uno, coge el que exista
        if file_do0 is None:
            best_files.append(file_do1)
            continue
        if file_do1 is None:
            best_files.append(file_do0)
            continue

        # si están los dos, compara val_loss
        best_files.append(file_do0 if get_val_loss(file_do0) < get_val_loss(file_do1) else file_do1)
    
    return best_files


In [6]:
def replace_with_symbols(file_name, mapping_dict):
    parts = file_name.split('_')
    #TODO
    parts = [parts[4], parts[2]+'_'+parts[3], parts[0]+'_'+parts[1] ]
    modified_parts = []
    for part in parts:
        if part in mapping_dict:
            modified_parts.append(mapping_dict[part])
    sorted_modified_parts = [modified_parts[0]] + sorted(modified_parts[-2:])
    modified_name = '+'.join(sorted_modified_parts)
    return modified_name

In [7]:
mapping_dict = {
    'phi_h': r'$\phi_h$',
    'phi_l': r'$\phi_\ell$',
    'phi_f': r'$\phi_f$',
    'phi_A': r'$\phi_A$',
    'Phi_H': r'$\Phi_H$',
    'Phi_cm': r'$\Phi_{\Sigma}$',
    'Phi_mu': r'$\Phi_{\mu}$',
    'Phi_G': r'$\Phi_G$',
    'L2': r'$\mathcal{O}_M$',
    'LT': r'$\mathcal{O}_T$',
    'LMq': r'$\mathcal{O}_R$',
    'LZ': r'$\mathcal{O}_D$',
}

### With seeds

In [ ]:
direc = ['reports_0/', 'reports_909/', 'reports_1234/'] #, 'reports_2032/']
seed = ['0', '909', '1234'] #, '2032']
for d in datasets:
    results = {}
    for i, directory in enumerate(direc):
        dir = directory + d
        files_list = compare_files(dir)
        files_list = sorted(files_list)
        
        for file_path in files_list:  
            with open(file_path, 'r') as f:
                report = json.load(f)
            file = file_path.split('/')[-1:][0]
            if file.endswith('dropout'):
                file = file[:-8]
            df = pd.read_json(file_path)
            aes = df["mae_test"].values
            if file not in results:
                results[file] = {}
            results[file][seed[i]] = aes

    for k, r in enumerate(results):
        ae = np.mean(list(results[r].values()), axis=0)
        method_name = replace_with_symbols(r, mapping_dict)
        table_ae.add(benchmark=d, method=method_name, v=ae)

methods = table_ae.get_methods()                        
methods_replace = {m: m for m in methods}
Table.LatexPDF(f'./latex/table_phiNN_big_seeds_def.pdf', [table_ae], dedicated_pages=False, transpose=True, method_replace=methods_replace)


currently in /media/nas/olayap/env_olaya/Doctorado/representation_LQ
[Tables Done] runing latex
This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./table_phiNN_big_seeds_def__.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/base/inputenc.sty)
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsmath.sty
For additional information on amsmath, use the `?' option.
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amstext.sty
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsgen.sty))
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsbsy.sty)
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsopn.sty))
(/usr/share/texlive/texmf-dist

### Without seeds

In [9]:
seed = '0' #, '909', '1234', '2032']
direc = [f'results/reports_{seed}/'] #, 'reports_909/', 'reports_1234/', 'reports_2032/']
table_rae_seed = Table(name='mrae')
for d in datasets:
    results = {}
    for i, directory in enumerate(direc):
        dir = directory + d
        print(dir)
        files_list = compare_files(dir)
        files_list = sorted(files_list)
        
        for file_path in files_list:  
            with open(file_path, 'r') as f:
                report = json.load(f)
            file = file_path.split('/')[-1:][0]
            if file.endswith('dropout'):
                file = file[:-8]
            df = pd.read_json(file_path)
            aes = df["mrae_test"].values
            results[file] = aes
            
    for k, r in enumerate(results):
        method_name = replace_with_symbols(r, mapping_dict)
        table_rae_seed.add(benchmark=d, method=method_name, v=results[r])

methods = table_rae_seed.get_methods()                        
methods_replace = {m: m for m in methods}
Table.LatexPDF(f'./latex/table_phiNN_big_def_mean_RAE_LEQUA_.pdf', [table_rae_seed], dedicated_pages=False, transpose=True, method_replace=methods_replace)


results/reports_0/T1B
results/reports_0/T2
currently in /media/nas/olayap/env_olaya/Doctorado/representation_LQ
[Tables Done] runing latex
[Done]


sh: 1: pdflatex: not found
rm: no se puede borrar 'table_phiNN_big_def_mean_RAE_LEQUA_.aux': No existe el archivo o el directorio
rm: no se puede borrar 'table_phiNN_big_def_mean_RAE_LEQUA_.log': No existe el archivo o el directorio


## Best models

In [ ]:
seed = '0' #, '909', '1234', '2032']
direc = [f'reports_{seed}/'] #, 'reports_909/', 'reports_1234/', 'reports_2032/']
table_ae_seed = Table(name='mae')

best = ['phi_h_Phi_H_LMq',
        'phi_l_Phi_H_LMq',
        'phi_h_Phi_cm_LMq',
        'phi_h_Phi_mu_L2',
        'phi_h_Phi_H_LT',
        'phi_l_Phi_H_LT',
        'phi_f_Phi_H_LT',
        #'phi_A_Phi_H_LZ', #This is the best phi_A
        'phi_h_Phi_H_LZ',
        'phi_l_Phi_H_LZ',
        'phi_f_Phi_H_LZ']
for d in datasets:
    results = {}
    for i, directory in enumerate(direc):
        dir = directory + d
        files_list = compare_files(dir)
        files_list = sorted(files_list)
        
        for file_path in files_list:  
            with open(file_path, 'r') as f:
                report = json.load(f)
            file = file_path.split('/')[-1:][0]
            if file.endswith('dropout'):
                file = file[:-8]
            if file in best:    
                df = pd.read_json(file_path)
                aes = df["mae_test"].values
                results[file] = aes
            else:
                continue
            
    for k, r in enumerate(results):
        method_name = replace_with_symbols(r, mapping_dict)
        table_ae_seed.add(benchmark=d, method=method_name, v=results[r])

methods = table_ae_seed.get_methods()                        
methods_replace = {m: m for m in methods}
Table.LatexPDF(f'./latex/table_phiNN_best_PhiLoss.pdf', [table_ae_seed], dedicated_pages=False, transpose=True, method_replace=methods_replace)

currently in /media/nas/olayap/env_olaya/Doctorado/representation_LQ
[Tables Done] runing latex
This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./table_phiNN_best_PhiLoss.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/base/inputenc.sty)
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsmath.sty
For additional information on amsmath, use the `?' option.
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amstext.sty
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsgen.sty))
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsbsy.sty)
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsopn.sty))
(/usr/share/texlive/texmf-dist/te

# Sankey Diagram

In [15]:
data = []

for method in table_ae.get_methods():
    m = method.split('+')
    values = table_ae.get_method_values(method)
    for i, value in enumerate(values):
        data.append({'Benchmark': table_ae.get_benchmarks()[i], 'Loss': m[0], 'Phi': m[1], 'phi': m[2], 'Value': value})

df_data = pd.DataFrame(data)

In [16]:
latex_to_utf8 = {
    "$\\phi_A$": "𝝓A",
    "$\\phi_\\ell$": "𝝓ℓ",
    "$\\phi_f$": "𝝓𝑓",
    "$\\phi_h$": "𝝓h",
    "$\\Phi_H$": "𝜱ʜ",
    "$\\Phi_{\\mu}$": "𝜱μ",
    "$\\Phi_{\\Sigma}$": "𝜱Σ",
    "$\\mathcal{O}_T$": "𝓞ᴛ",
    "$\\mathcal{O}_D$": "𝓞ᴅ",
    "$\\mathcal{O}_R$": "𝓞ʀ",
    "$\\mathcal{O}_M$": "𝓞ᴍ" 
}

def convert_latex_to_utf8(df, columns):
    for col in columns:
        df[col] = df[col].replace(latex_to_utf8)
    return df

In [17]:
def hex_to_rgb(color):
    return np.array(mcolors.to_rgb(color)) * 255

def link_color(col1, col2):
    rgb1 = hex_to_rgb(col1)
    rgb2 = hex_to_rgb(col2)
    color = np.sqrt((rgb1**2 + rgb2**2) / 2)
    return "rgba(%d, %d, %d, 0.5)" % tuple(color)

### Top n

In [23]:
n = 3

In [24]:
top_n_per_benchmark = df_data.groupby('Benchmark').apply(lambda x: x.nsmallest(n, 'Value')).reset_index(drop=True)
df_combined = top_n_per_benchmark.groupby(['phi', 'Phi', 'Loss']).size().reset_index(name='Count')

top_n_per_benchmark['Ranking'] = top_n_per_benchmark.groupby('Benchmark')['Value'].rank(method='first')
top_n_per_benchmark['Score'] = 1 /top_n_per_benchmark['Ranking'] *100 #np.log2(top_n_per_benchmark['Ranking'] + 1) <- ifS DGC Scre

dcg_weights = top_n_per_benchmark.groupby(['phi', 'Phi', 'Loss'])['Score'].sum().reset_index()

df_combined = df_combined.merge(dcg_weights, on=['phi', 'Phi', 'Loss'], how='left')

/tmp/ipykernel_1903228/2448597990.py:1: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [25]:
df = df_combined.sort_values(by=['phi', 'Phi', 'Loss']).reset_index(drop=True)
df = convert_latex_to_utf8(df, ['phi', 'Phi', 'Loss'])

In [26]:
def mix_colors(colors, num, total):
    r_total, g_total, b_total = 0, 0, 0
    for color, porcentaje in zip(colors, num):
        rgb_values = color[color.find('(')+1:color.find(')')].split(',')
        r, g, b = int(rgb_values[0]), int(rgb_values[1]), int(rgb_values[2])        
        r_total += r * (porcentaje / total)
        g_total += g * (porcentaje / total)
        b_total += b * (porcentaje / total)
    
    r_final = round(r_total)
    g_final = round(g_total)
    b_final = round(b_total)
    
    return f"rgb({r_final}, {g_final}, {b_final})"


In [27]:
phis = df['phi'].unique()

phi_colors = {'𝝓A': 'rgb(248, 156, 116)', #red
            '𝝓ℓ':  'rgb(246, 207, 113)', #yellow
            '𝝓h': 'rgb(102, 197, 204)', #blue
            '𝝓𝑓': 'rgb(220, 176, 242)'} #purple

all_labels = pd.concat([df['phi'], df['Phi'], df['Loss']]).unique()
label_indices = {label: idx for idx, label in enumerate(all_labels)}

node_colors = [phi_colors.get(label, 'rgb(209, 208, 207)') for label in all_labels] 

node_color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(all_labels)}

def rgb_to_rgba(color, alpha):
    if isinstance(color, str): 
        rgb_values = color[color.find('(')+1:color.find(')')].split(',')
        r, g, b = int(rgb_values[0]), int(rgb_values[1]), int(rgb_values[2])
    else: 
        r, g, b = color
    return f'rgba({r}, {g}, {b}, {alpha})'

node_rgba_colors = [rgb_to_rgba(node_color_map[label], 0.9) for label in all_labels]

sources = []
targets = []
values = []
link_colors = []
link_opacity = 0.4 #transparency

for _, row in df.iterrows():
    phi_color = node_color_map[row['phi']]
    
    rgba_color = rgb_to_rgba(phi_color, link_opacity)
    
    # phi -> Phi link
    sources.append(label_indices[row['phi']])
    targets.append(label_indices[row['Phi']])
    values.append(row['Score'])
    link_colors.append(rgba_color)  
    
    # Phi -> Loss link
    sources.append(label_indices[row['Phi']])
    targets.append(label_indices[row['Loss']])
    values.append(row['Score'])
    link_colors.append(rgba_color)  

sankey = go.Sankey(
    node=dict(
        pad=30,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=list(all_labels),
        color=node_rgba_colors,  
        hovertemplate="%{label}<extra></extra>"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,  
        hovertemplate="Flow: %{value}<extra></extra>"
    )
)

layout = go.Layout(
    title_text=f"Top {n} Flows",
    font_size=12,
    font_color="black",
    title_font_color="darkblue",
    title_font_size=18,
    font_family="Arial",
    showlegend=False,
    plot_bgcolor="rgba(0,0,0,0)",  
    paper_bgcolor="white",  
)

fig = go.Figure(data=[sankey], layout=layout)

fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)

fig.show()
fig.write_image(f"./sankey_diagrams/sankey_diag_top{n}.pdf")